# Projet Tinder : qu'est-ce qui transforme une rencontre en match ?


**Contexte**  
L'équipe marketing de Tinder observe une baisse du nombre de matchs sur la plateforme.   
Elle exploite une expérience menée entre 2002 et 2004 : des étudiants ont participé à des speed datings de 4 minutes,  
puis ont indiqué en secret s'ils souhaitaient revoir chaque partenaire.   
Un **match** a lieu lorsque les deux partenaires répondent « oui ».

**Données**  
'data/dataset.csv'

**Etapes du projet**
1. Imports et configuration
2. Découverte du jeu de données
3. Nettoyage et constitution des tables d'analyse
4. Profil des participants (analyse univariée)
5. Facteurs associés aux décisions (analyses descriptives et comparatives)
6. Quelques tests statistiques simples
7. Synthèse, recommandations et limites


**Questions explorées**

| # | Question | 
|---|---|
| Q1 | Quelle proportion de rencontres aboutit à un match ? Existe-t-il une différence entre les sexes ? |
| Q2 | Quels critères distinguent les rencontres qui plaisent ? | 
| Q3 | Les préférences déclarées correspondent-elles aux perceptions de l'autre sexe ? |
| Q4 | Les participants savent-ils évaluer leur propre valeur ? |
| Q5 | Les intérêts communs et l'origine commune favorisent-ils les matchs ? |
| Q6 | Le moment de la rencontre influence-t-il la décision ? |

## 1. Imports et configuration

Les bibliothèques utilisées sont `pandas` et `numpy` pour les données, `plotly` pour les graphiques et `scipy` pour les tests statistiques simples. Le thème Plotly reprend les couleurs de Tinder.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats


pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

In [2]:
# Palette Tinder
TINDER_PINK = "#FD3476"
TINDER_RED = "#FE5764"
TINDER_ORANGE = "#FF7854"
TINDER_PEACH = "#FFA07A"
GENDER_COLORS = {"Femme": TINDER_PINK, "Homme": TINDER_PEACH}

# Thème Plotly : fond sombre, couleurs Tinder
tinder_template = go.layout.Template()
tinder_template.layout = go.Layout(
    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FFFFFF",
    font=dict(color="black", family="Arial", size=14),
    title=dict(font=dict(size=20, color="black"), x=0.5),
    xaxis=dict(gridcolor="#2A2A2A", linecolor="#444444"),
    yaxis=dict(gridcolor="#2A2A2A", linecolor="#444444"),
    legend=dict(font=dict(size=13, color="black"), bgcolor="rgba(255,255,255,255)"),
    colorway=[TINDER_PINK, TINDER_ORANGE, TINDER_RED, TINDER_PEACH],
)
pio.templates["tinder_dark"] = tinder_template
pio.templates.default = "tinder_dark"

In [3]:
DATA_PATH = Path("../data") / "dataset.csv"
IMG_PATH = Path ("../assets/images")
IMG_PATH.mkdir(parents=True, exist_ok=True)

In [4]:
def count_bar(df, col, title, xaxis_title, by=None, category_order=None):
    """Diagramme en barres des effectifs d'une variable catégorielle."""
    group_cols = [col] if by is None else [col, by]
    counts = df.groupby(group_cols).size().reset_index(name="n")
    fig = px.bar(counts, x=col, y="n", color=by, barmode="group", text="n",
                 title=title, color_discrete_map=GENDER_COLORS if by else None,
                 category_orders={col: category_order} if category_order else None)
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.update_layout(xaxis_title=xaxis_title, yaxis_title="Nombre de participants",
                      bargap=0.2, height=450, width=850)
    return fig

def normalize_to_100(df, cols):
    """Ramène à 100 les points répartis entre plusieurs critères."""
    total = df[cols].sum(axis=1, min_count=len(cols))
    return df[cols].div(total.where(total > 0), axis=0) * 100


## 2. Découverte du jeu de données

In [5]:
df = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")
n_part = df["iid"].nunique()
print(f"{df.shape[0]} lignes, {df.shape[1]} colonnes, {n_part} participants")
df.iloc[:3, :16]

8378 lignes, 195 colonnes, 551 participants


,iid,id,gender,idg,condtn,wave,round,position,positin1,order,partner,pid,match,int_corr,samerace,age_o
0,1,1.0,0,1,1,1,10,7,NaN,4,1,11.0,0,0.14,0,27.0
1,1,1.0,0,1,1,1,10,7,NaN,3,2,12.0,0,0.54,0,22.0
2,1,1.0,0,1,1,1,10,7,NaN,10,3,13.0,1,0.16,1,22.0


In [6]:
KEY_COLS = ["age", "match", "dec", "attr", "sinc", "intel", "fun", "amb",
            "shar", "like", "prob"]

df[KEY_COLS].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,8283.0,26.36,3.57,18.0,24.0,26.0,28.0,55.0
match,8378.0,0.16,0.37,0.0,0.0,0.0,0.0,1.0
dec,8378.0,0.42,0.49,0.0,0.0,0.0,1.0,1.0
attr,8176.0,6.19,1.95,0.0,5.0,6.0,8.0,10.0
sinc,8101.0,7.18,1.74,0.0,6.0,7.0,8.0,10.0
intel,8082.0,7.37,1.55,0.0,6.0,7.0,8.0,10.0
fun,8028.0,6.40,1.95,0.0,5.0,7.0,8.0,10.0
amb,7666.0,6.78,1.79,0.0,6.0,7.0,8.0,10.0
shar,7311.0,5.47,2.16,0.0,4.0,6.0,7.0,10.0
like,8138.0,6.13,1.84,0.0,5.0,6.0,7.0,10.0


In [7]:
missing_rate = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
nb_missing = (missing_rate > 20).sum()

print(f"{nb_missing} colonnes sur {df.shape[1]} dépassent 20 % de valeurs manquantes")

missing_rate.head(8)

90 colonnes sur 195 dépassent 20 % de valeurs manquantes


num_in_3    92.0
numdat_3    82.1
expnum      78.5
sinc7_2     76.7
amb7_2      76.7
shar7_2     76.4
attr7_2     76.3
fun7_2      76.3
dtype: float64

Le fichier contient une ligne par rencontre **vue du participant**. Chaque rencontre apparaît donc **deux fois** (une fois par partenaire). Les 195 colonnes se répartissent en :

- **identifiants et caractéristiques de la rencontre** (participant, partenaire, vague, ordre de passage, `match`, `samerace`...) ;
- **questionnaire d'inscription (T1)** : profil, loisirs et préférences déclarées, répétés à chaque rencontre d'un même participant ;
- **scorecard** : décision (`dec`) et notes données au partenaire juste après la rencontre ;
- **questionnaires de suivi (T2 et T3)**, remplis le lendemain et un mois plus tard.

Les notes sont sur une échelle de 1 à 10. Les préférences sont réparties en 100 points entre six critères : attirance, sincérité, intelligence, humour, ambition et intérêts communs.

**Observations**
- 8 378 rencontres, 551 participants, 195 colonnes.
- 90 colonnes dépassent 20 % de valeurs manquantes. Elles viennent surtout des questionnaires T2 et T3 (le taux de réponse chute après la soirée) et de questions posées seulement dans certaines vagues.
- Le taux de décisions positives est de 42 % ; le taux de match de 16 %.
- Les notes vont de 0 à 10 : quelques valeurs hors échelle sont à contrôler.

## 3. Nettoyage et constitution des tables d'analyse

### 3.1 Choix du périmètre

**Vagues 6 à 9 : exclusion complète.** Selon la documentation, les participants de ces vagues ont noté l'importance des critères sur une échelle de 1 à 10, et non en répartissant 100 points. Aucune documentation n'indique comment ces notes ont été retraitées. Pour garder une mesure homogène, ces vagues sont exclues de toute l'analyse. Le coût est modéré : 102 participants sur 551 (18,5 %) et 1 562 rencontres sur 8 378 (18,6 %). Il reste 449 participants et 6 816 rencontres.

On retire ensuite les questionnaires de suivi T2 et T3 (hors du périmètre de la problématique) et les colonnes dont plus de 20 % des valeurs sont manquantes.

In [8]:
SCORE_COLS = ["attr", "sinc", "intel", "fun", "amb", "shar"]
SCORE_LABELS = ["Attirance", "Sincérité", "Intelligence", "Humour",
                "Ambition", "Intérêts communs"]
LABELS = dict(zip(SCORE_COLS, SCORE_LABELS))

# Vagues 6 à 9 : méthode de notation sur 1-10 au lieu de 100 points -> exclues
df_clean = df[~df["wave"].between(6, 9)].copy()

# Seuls les questionnaires d'inscription (T1) et la scorecard sont conservés
df_clean = df_clean.loc[:, :"amb3_s"].copy()

print(f"Lignes : {len(df)} -> {len(df_clean)} "
      f"({len(df) - len(df_clean)} rencontres des vagues 6 à 9 retirées)")
print(f"Participants : {df['iid'].nunique()} -> {df_clean['iid'].nunique()}")

Lignes : 8378 -> 6816 (1562 rencontres des vagues 6 à 9 retirées)
Participants : 551 -> 449


In [9]:
THRESHOLD = 0.20

missing_rate = df_clean.isna().mean()
cols_to_drop = missing_rate[missing_rate > THRESHOLD].index.tolist()
print(f"{len(cols_to_drop)} colonnes supprimées :")
print(cols_to_drop)

df_clean = df_clean.drop(columns=cols_to_drop)

28 colonnes supprimées :
['positin1', 'undergra', 'mn_sat', 'tuition', 'income', 'expnum', 'attr4_1', 'sinc4_1', 'intel4_1', 'fun4_1', 'amb4_1', 'shar4_1', 'attr5_1', 'sinc5_1', 'intel5_1', 'fun5_1', 'amb5_1', 'attr1_s', 'sinc1_s', 'intel1_s', 'fun1_s', 'amb1_s', 'shar1_s', 'attr3_s', 'sinc3_s', 'intel3_s', 'fun3_s', 'amb3_s']


**Observation** : les colonnes supprimées correspondent à la scolarité et au revenu (jamais ou rarement renseignés), aux questions sur « ce que recherchent les autres » (attr4_1...), à la perception des autres (attr5_1...) et aux questions posées en milieu de soirée (colonnes `_s`).

### 3.2 Contrôles de qualité

- La règle `match = dec × dec_o` est vérifiée sur toutes les lignes (`assert`).
- 2 notes reçues dépassent 10 (erreurs de saisie) : elles sont remplacées par des valeurs manquantes.
- 10 rencontres n'ont pas de partenaire identifié : elles sont retirées car elles empêchent de reconstituer les paires.



In [10]:
# 1) Cohérence : un match = les deux partenaires ont dit oui
assert (df_clean["match"] == df_clean["dec"] * df_clean["dec_o"]).all()

# 2) Notes hors de l'échelle 1-10 (erreurs de saisie) -> valeurs manquantes
RECEIVED_COLS = [f"{col}_o" for col in SCORE_COLS]
for cols in (SCORE_COLS, RECEIVED_COLS):
    nb_out = (df_clean[cols] > 10).sum().sum()
    df_clean[cols] = df_clean[cols].where(~(df_clean[cols] > 10))
    print(f"{nb_out} notes > 10 mises à NaN dans {cols[0]}...")

# 3) Rencontres sans identifiant de partenaire : inutilisables pour les paires
nb_no_pid = df_clean["pid"].isna().sum()
df_clean = df_clean.dropna(subset=["pid"])
print(f"{nb_no_pid} rencontres sans partenaire identifié retirées")

0 notes > 10 mises à NaN dans attr...
2 notes > 10 mises à NaN dans attr_o...
10 rencontres sans partenaire identifié retirées


### 3.3 Trois tables, trois unités d'analyse

Le choix de l'unité d'analyse est essentiel : compter deux fois une même rencontre altérerait les analyses.

| Table | Une ligne = | Sert à |
|---|---|---|
| `df_participants` | un participant | profil, préférences déclarées, auto-évaluation |
| `df_dates` | une rencontre vue d'un participant | notes données, décisions, ordre de passage |
| `df_pairs` | une rencontre (les deux partenaires) | match, origine commune, intérêts communs |

Les préférences réparties en 100 points ne totalisent pas toujours 100 (12 participants pour les préférences déclarées, de 90 à 148 ; 20 pour celles prêtées à l'autre sexe). Elles sont renormalisées à 100.

In [11]:
GENRE = {0: "Femme", 1: "Homme"}
DOMAINES = {
    1: "Droit", 2: "Maths", 3: "Sc. soc./Psycho", 4: "Médecine/Pharma",
    5: "Ingénierie", 6: "Anglais/Écriture", 7: "Histoire/Philo",
    8: "Éco/Finance", 9: "Éducation", 10: "Bio/Chimie/Physique",
    11: "Travail social", 12: "Indécis", 13: "Sc. politiques", 14: "Cinéma",
    15: "Arts", 16: "Langues", 17: "Architecture", 18: "Autre",
}
ORIGINE = {
    1: "Afro-américain", 2: "Européen/Caucasien", 3: "Latino", 4: "Asiatique",
    5: "Amérindien", 6: "Autre",
}
OBJECTIF = {
    1: "Soirée sympa", 2: "Rencontres", 3: "Trouver un date",
    4: "Relation sérieuse", 5: "Pour tester", 6: "Autre",
}
FREQUENCE = {
    1: "1 - Plusieurs/sem.", 2: "2 - 2x/sem.", 3: "3 - 1x/sem.",
    4: "4 - 2x/mois", 5: "5 - 1x/mois", 6: "6 - Plusieurs/an", 7: "7 - Jamais",
}
FREQ_ORDER = list(FREQUENCE.values())

In [12]:
PREF_COLS = {
    "declared": ["attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1"],
    "believed": ["attr2_1", "sinc2_1", "intel2_1", "fun2_1", "amb2_1", "shar2_1"],
}
SELF_COLS = ["attr3_1", "sinc3_1", "intel3_1", "fun3_1", "amb3_1"]
PROFILE_COLS = ["gender", "age", "field_cd", "race", "goal", "date", "go_out"]

# Une ligne par participant (les infos d'inscription sont répétées à chaque rencontre)
df_participants = df_clean.drop_duplicates(subset="iid")[
    ["iid"] + PROFILE_COLS + PREF_COLS["declared"] + PREF_COLS["believed"] + SELF_COLS
].copy()
assert df_participants["iid"].is_unique

nb_before = df_participants.shape[0]
df_participants = df_participants.dropna(subset=PROFILE_COLS)
print(f"{nb_before - df_participants.shape[0]} participants sans profil complet retirés "
      f"-> {df_participants.shape[0]} participants")

# Libellés
df_participants["gender"] = df_participants["gender"].map(GENRE)
df_participants["field_cd"] = df_participants["field_cd"].astype("Int64").map(DOMAINES)
df_participants["race"] = df_participants["race"].astype("Int64").map(ORIGINE)
df_participants["goal"] = df_participants["goal"].astype("Int64").map(OBJECTIF)
df_participants["date"] = df_participants["date"].astype("Int64").map(FREQUENCE)
df_participants["go_out"] = df_participants["go_out"].astype("Int64").map(FREQUENCE)

# Préférences : somme doit valoir 100 -> contrôle puis renormalisation
for key in ("declared", "believed"):
    cols = PREF_COLS[key]
    totals = df_participants[cols].sum(axis=1, min_count=6)
    nb_off = (totals.round() != 100).sum()
    print(f"{key} : {nb_off} participants dont la somme n'est pas 100 "
          f"(min {totals.min():.0f}, max {totals.max():.0f}) -> renormalisés")
    df_participants[cols] = normalize_to_100(df_participants, cols)
df_participants.head(3)

10 participants sans profil complet retirés -> 439 participants
declared : 12 participants dont la somme n'est pas 100 (min 90, max 148) -> renormalisés
believed : 20 participants dont la somme n'est pas 100 (min 80, max 112) -> renormalisés


,iid,gender,age,field_cd,race,goal,date,go_out,attr1_1,sinc1_1,intel1_1,fun1_1,amb1_1,shar1_1,attr2_1,sinc2_1,intel2_1,fun2_1,amb2_1,shar2_1,attr3_1,sinc3_1,intel3_1,fun3_1,amb3_1
0,1,Femme,21.0,Droit,Asiatique,Rencontres,7 - Jamais,1 - Plusieurs/sem.,15.0,20.0,20.0,15.0,15.0,15.0,35.0,20.0,15.0,20.0,5.0,5.0,6.0,8.0,8.0,8.0,7.0
10,2,Femme,24.0,Droit,Européen/Caucasien,Soirée sympa,5 - 1x/mois,1 - Plusieurs/sem.,45.0,5.0,25.0,20.0,0.0,5.0,65.0,0.0,10.0,25.0,0.0,0.0,7.0,5.0,8.0,10.0,3.0
20,3,Femme,25.0,Maths,Européen/Caucasien,Autre,3 - 1x/sem.,1 - Plusieurs/sem.,35.0,10.0,35.0,10.0,10.0,0.0,50.0,0.0,20.0,30.0,0.0,0.0,8.0,9.0,9.0,8.0,8.0


In [13]:
DATE_COLS = (
    ["iid", "pid", "wave", "gender", "round", "order", "match", "dec", "dec_o",
     "samerace", "int_corr", "like", "prob"]
    + SCORE_COLS + RECEIVED_COLS
)

# Une ligne par rencontre, vue du participant (chaque rencontre apparaît 2 fois)
df_dates = df_clean[DATE_COLS].copy()
df_dates["gender"] = df_dates["gender"].map(GENRE)

# Identifiant de paire, identique pour les deux participants d'une même rencontre
ids = df_dates[["iid", "pid"]].astype(int)
df_dates["pair_id"] = ids.min(axis=1).astype(str) + "_" + ids.max(axis=1).astype(str)

# Une ligne par paire (pour les variables communes aux deux partenaires)
df_pairs = df_dates.drop_duplicates(subset="pair_id")[
    ["pair_id", "wave", "match", "samerace", "int_corr"]
].copy()

print(f"df_dates : {len(df_dates)} lignes | df_pairs : {len(df_pairs)} paires")
print(f"Les {len(df_dates)} lignes correspondent à "
      f"{df_dates['pair_id'].nunique()} paires")

df_dates : 6806 lignes | df_pairs : 3403 paires
Les 6806 lignes correspondent à 3403 paires


In [14]:
# Lignes retenues pour les analyses sur les notes : les 6 notes sont renseignées
has_scores = df_dates[SCORE_COLS].notna().all(axis=1)
df_scored = df_dates[has_scores].copy()

print(f"{(~has_scores).sum()} rencontres sans les 6 notes écartées "
      f"({(~has_scores).mean():.1%}) -> {len(df_scored)} rencontres analysées")
print("Taux de décision positive, rencontres écartées vs conservées : "
      f"{df_dates.loc[~has_scores, 'dec'].mean():.1%} vs {df_scored['dec'].mean():.1%}")

1064 rencontres sans les 6 notes écartées (15.6%) -> 5742 rencontres analysées
Taux de décision positive, rencontres écartées vs conservées : 36.6% vs 42.7%


**Observation** : 15,6 % des rencontres n'ont pas les six notes. Ces rencontres écartées ont un taux de « oui » plus faible (36,6 % contre 42,7 %). C'est un biais possible à garder en tête, qui est repris dans les limites (section 7.3).

## 4. Profil des participants (analyse univariée)

Les informations d'inscription sont répétées à chaque rencontre : cette partie utilise `df_participants` (une ligne par participant, 439 participants avec un profil complet).

In [15]:
print(df_participants["gender"].value_counts().to_frame("participants").T)
print()
print(df_participants.groupby("gender")["age"].describe().round(1))

gender        Homme  Femme
participants    221    218

        count  mean  std   min   25%   50%   75%   max
gender                                                
Femme   218.0  26.1  4.2  19.0  23.0  25.5  28.0  55.0
Homme   221.0  26.5  3.5  18.0  24.0  27.0  29.0  39.0


In [16]:
age_counts = df_participants.groupby(["age", "gender"]).size().reset_index(name="nb")
fig = px.bar(
    age_counts, x="age", y="nb", color="gender", barmode="stack",
    color_discrete_map=GENDER_COLORS,
    title="Une population jeune : la majorité a entre 23 et 30 ans",
)

fig.update_layout(xaxis_title="Âge", yaxis_title="Nombre de participants",
                  height=420, width=850)

fig.show()

fig.write_image(IMG_PATH / "01_age_participants.png")

age_mean = df_participants['age'].mean().round(1)
print(f"Âge moyen des participants : {age_mean}")

Âge moyen des participants : 26.3


In [17]:
field_order = df_participants["field_cd"].value_counts().index.tolist()

fig = count_bar(
    df_participants, "field_cd",
    "Domaine d'études : une forte présence de l'économie et de la finance",
    "Domaine d'études", by="gender", category_order=field_order,
)

fig.update_layout(xaxis_tickangle=-40, height=550, width=1000, margin=dict(b=150))

fig.show()

fig.write_image(IMG_PATH / "02_domaine_d_etudes.png")

pct_finance = (df_participants["field_cd"]=="Éco/Finance").sum() / df_participants.shape[0] * 100
print(f"{pct_finance.round(0)}% des participants en éco/finances")

21.0% des participants en éco/finances


In [18]:
goal_order = df_participants["goal"].value_counts().index.tolist()
fig = count_bar(
    df_participants, "goal",
    "Peu de participants recherchaient une relation sérieuse",
    "Objectif déclaré", by="gender", category_order=goal_order,
)

fig.update_layout(xaxis_tickangle=-30, margin=dict(b=120))

fig.show()

fig.write_image(IMG_PATH / "03_objectif_participants.png")

In [19]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("Fréquence des dates", "Fréquence des sorties"))

for col_idx, col in enumerate(["date", "go_out"], start=1):
    counts = df_participants.groupby([col, "gender"]).size().reset_index(name="nb")
    for gender, color in GENDER_COLORS.items():
        sub = counts[counts["gender"] == gender].set_index(col).reindex(FREQ_ORDER)
        fig.add_trace(
            go.Bar(x=sub.index, y=sub["nb"].fillna(0), name=gender,
                   marker_color=color, showlegend=(col_idx == 1)),
            row=1, col=col_idx,
        )

fig.update_layout(barmode="group", height=480, width=1000,
                  title="Des habitués des sorties, mais pas des dates",
                  margin=dict(b=120))

fig.update_xaxes(tickangle=-40)

fig.update_yaxes(title_text="Nombre de participants", col=1)

fig.show()

fig.write_image(IMG_PATH / "04_habitudes_participants.png")

**Ce qu'il faut retenir**
- **Une population jeune et mixte** : 218 femmes et 221 hommes, 26,3 ans en moyenne, majoritairement entre 23 et 30 ans.
- **Une forte présence de l'économie et de la finance** (21 % des participants), très genrée : 36 % des hommes contre 7 % des femmes.
- **Peu viennent chercher l'amour** : 76 % ont répondu « soirée sympa » ou « rencontrer des gens ». Seuls 11 % visaient un date ou une relation sérieuse, davantage chez les hommes (11 % pour « trouver un date » contre 4 % chez les femmes).
- **Des habitués des sorties, mais pas des dates** : 90 % sortent au moins une fois par semaine, alors que 60 % déclarent avoir un date au plus une fois par mois.
- **Origines** : 55 % de Caucasiens et 25 % d'Asiatiques.

**Conséquence** : l'échantillon est composé d'étudiants (Columbia), jeunes et diplômés. Les résultats ne se généralisent pas tels quels à l'ensemble des utilisateurs de Tinder.

## 5. Facteurs associés aux décisions (analyses descriptives et comparatives)

### 5.1 Combien de matchs, et qui dit oui ?

In [20]:
# Taux de match : une ligne par paire
match_rate = df_pairs["match"].mean()
print(f"Taux de match : {match_rate:.1%} ({int(df_pairs['match'].sum())} matchs sur {len(df_pairs)} paires)")

# Taux de décision positive : une ligne par participant et par rencontre
dec_by_gender = df_dates.groupby("gender")["dec"].mean().rename("taux_oui").to_frame()
print((100 * dec_by_gender).round(1).rename(columns={"taux_oui": "% de oui"}))

fig = go.Figure(go.Bar(x=dec_by_gender.index, y=dec_by_gender["taux_oui"] * 100,
    marker_color=[GENDER_COLORS[gender] for gender in dec_by_gender.index],
    text=(dec_by_gender["taux_oui"] * 100).round(1).astype(str) + " %",
    textposition="inside"))
fig.update_layout(title="Les hommes disent plus souvent oui que les femmes",
    yaxis_title="Décisions positives (%)", xaxis_title="Sexe", height=430, width=800,
    yaxis_range=[0, 60])
fig.show()

fig.write_image(IMG_PATH / "05_taux_decision_positive.png")

Taux de match : 16.5% (562 matchs sur 3403 paires)
        % de oui
gender          
Femme       37.9
Homme       45.6


**Interprétation**
- Environ **une rencontre sur six débouche sur un match**.
- Dans cet échantillon, les hommes disent plus souvent « oui » que les femmes.
- Ces résultats sont descriptifs et ne prouvent pas à eux seuls une relation de cause à effet.

### 5.2 Les notes données selon la décision

In [21]:
mean_scores = (
    df_scored.groupby("dec")[SCORE_COLS].mean().T
    .rename(columns={0: "Non", 1: "Oui"})
    .rename(index=LABELS)
)
mean_scores["Écart"] = mean_scores["Oui"] - mean_scores["Non"]
mean_scores = mean_scores.sort_values("Écart", ascending=False)
print(mean_scores.round(2))

fig = go.Figure()
for answer, color in (("Non", "#666666"), ("Oui", TINDER_PINK)):
    fig.add_trace(go.Bar(
        x=mean_scores.index, y=mean_scores[answer], name=f"Décision : {answer}",
        marker_color=color, text=mean_scores[answer].round(1),
        textposition="outside", cliponaxis=False,
    ))
fig.update_layout(
    barmode="group", height=470, width=900,
    title="Un « oui » s'accompagne de notes plus hautes sur tous les critères",
    yaxis=dict(title="Note moyenne attribuée (1-10)", range=[0, 9.5]),
)
fig.show()

fig.write_image(IMG_PATH / "06_notes_selon_decision.png")

dec                Non   Oui  Écart
Attirance         5.35  7.24   1.89
Intérêts communs  4.64  6.44   1.80
Humour            5.66  7.31   1.66
Sincérité         6.81  7.56   0.75
Intelligence      7.03  7.72   0.70
Ambition          6.44  7.11   0.66


**Interprétation** : quand un participant dit « oui », il a noté son partenaire plus haut sur les six critères. L'écart est **grand pour l'attirance (+1,9 point), les intérêts communs (+1,8) et l'humour (+1,7)**, et **plus faible pour la sincérité (+0,8), l'intelligence (+0,7) et l'ambition (+0,7)**. Les critères « sérieux » distinguent peu les rencontres qui plaisent de celles qui ne plaisent pas.

On compare simplement les préférences déclarées à l'inscription. Ces réponses indiquent ce que les participants disent rechercher ; elles ne mesurent pas directement ce qui cause un match.


In [22]:
declared = df_participants.groupby("gender")[PREF_COLS["declared"]].mean()
declared.columns = SCORE_LABELS
fig = go.Figure()

for gender in ("Femme", "Homme"):
    fig.add_trace(go.Bar(x=SCORE_LABELS, y=declared.loc[gender], name=gender,
                         marker_color=GENDER_COLORS[gender]))

fig.update_layout(barmode="group", height=450, width=900,
                  title="Les critères déclarés à l'inscription",
                  xaxis_title="Critère", yaxis_title="Part moyenne déclarée (points sur 100)")

fig.show()

fig.write_image(IMG_PATH / "07_criteres_declares.png")


**Interprétation** : l'attirance est particulièrement importante dans les préférences déclarées, notamment chez les hommes. Il faut distinguer ces déclarations des comportements observés pendant les rencontres.

### 5.4 Ce que chaque sexe croit que l'autre recherche

In [23]:
# Ce que chaque sexe déclare chercher vs ce que l'autre sexe croit qu'il cherche
declared_means = df_participants.groupby("gender")[PREF_COLS["declared"]].mean()
believed_means = df_participants.groupby("gender")[PREF_COLS["believed"]].mean()
declared_means.columns = believed_means.columns = SCORE_LABELS

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("Ce que les FEMMES recherchent",
                                    "Ce que les HOMMES recherchent"))

for col_idx, (target, guesser) in enumerate((("Femme", "Homme"), ("Homme", "Femme")),
                                            start=1):
    fig.add_trace(go.Bar(x=SCORE_LABELS, y=declared_means.loc[target],
                         name="Déclaré par eux-mêmes", marker_color="#888888",
                         showlegend=(col_idx == 1)), row=1, col=col_idx)
    fig.add_trace(go.Bar(x=SCORE_LABELS, y=believed_means.loc[guesser],
                         name="Perçu par l'autre sexe",
                         marker_color=GENDER_COLORS[guesser],
                         showlegend=(col_idx == 1)), row=1, col=col_idx)

fig.update_layout(barmode="group", height=480, width=1000,
                  title="Chaque sexe surestime le poids de l'attirance chez l'autre")

fig.update_yaxes(title_text="Points sur 100", col=1)

fig.update_xaxes(tickangle=-30)

fig.show()

fig.write_image(IMG_PATH / "08_criteres_vus_par_autre_sexe.png")

**Interprétation** : **chaque sexe surestime le poids de l'attirance chez l'autre**. Les femmes déclarent lui accorder 18,7 points, mais les hommes leur en prêtent 26,7 (+8,1). Les hommes déclarent 29,2 points, mais les femmes leur en prêtent 38,6 (+9,4). Les deux sexes surestiment l'importance de l'attirance physique dans les préférences déclarées de l'autre sexe..

### 5.5 Peut-on estimer sa propre valeur ?

On compare l'auto-évaluation de chaque participant (inscription) à la note moyenne que lui ont donnée ses partenaires pendant la soirée.

In [24]:
SELF_MAP = {"attr3_1": "attr", "sinc3_1": "sinc", "intel3_1": "intel",
            "fun3_1": "fun", "amb3_1": "amb"}
self_cols = list(SELF_MAP.values())

# Note moyenne reçue par chaque participant (notes données par ses partenaires)
received = df_dates.groupby("iid")[RECEIVED_COLS].mean()
received.columns = SCORE_COLS

df_self = (
    df_participants.set_index("iid")[list(SELF_MAP)].rename(columns=SELF_MAP)
    .join(received, lsuffix="_self", rsuffix="_recu")
    .dropna()
)

print(f"{len(df_self)} participants avec auto-évaluation et notes reçues")

compare = pd.DataFrame({
    "Auto-évaluation": [df_self[f"{c}_self"].mean() for c in self_cols],
    "Note moyenne reçue": [df_self[f"{c}_recu"].mean() for c in self_cols],
}, index=[LABELS[c] for c in self_cols])

compare["Écart"] = compare["Auto-évaluation"] - compare["Note moyenne reçue"]

print(compare.round(2))

fig = go.Figure()
for col, color in (("Auto-évaluation", "#888888"), ("Note moyenne reçue", TINDER_PINK)):
    fig.add_trace(go.Bar(x=compare.index, y=compare[col], name=col,
                         marker_color=color, text=compare[col].round(1),
                         textposition="outside", cliponaxis=False))

fig.update_layout(barmode="group", height=450, width=850,
                  title="Auto-évaluation et notes moyennes reçues : comment se perçoit-on ?",
                  yaxis=dict(title="Note moyenne (1-10)", range=[0, 10]))

fig.show()

fig.write_image(IMG_PATH / "09_auto_evaluation.png")

# Peut-on prédire sa valeur ? Note reçue moyenne selon l'auto-évaluation (attirance)
bins = pd.cut(df_self["attr_self"], [0, 5, 6, 7, 8, 10],
              labels=["≤ 5", "6", "7", "8", "9-10"])

by_self = df_self.groupby(bins, observed=True)["attr_recu"].agg(["mean", "count"])

print(by_self.round(2))

437 participants avec auto-évaluation et notes reçues
              Auto-évaluation  Note moyenne reçue  Écart
Attirance                7.02                6.16   0.85
Sincérité                8.24                7.19   1.05
Intelligence             8.33                7.38   0.95
Humour                   7.68                6.40   1.27
Ambition                 7.53                6.78   0.75


           mean  count
attr_self             
≤ 5        5.56     58
6          6.04     66
7          6.11    149
8          6.33    118
9-10       6.84     46


**Interprétation**
- **Les participants s'évaluent plus favorablement que leurs partenaires ne les évaluent**, en moyenne, sur les cinq critères étudiés. L'écart varie de +0,75 point pour l'ambition à +1,27 point pour l'humour. Pour l'attirance, la note moyenne attribuée à soi-même est de 7,02, contre 6,16 pour la note moyenne reçue.
- **L'auto-évaluation de l'attirance semble néanmoins refléter partiellement la perception des autres** : Les participants qui s'attribuent les meilleures notes reçoivent également les meilleures notes en moyenne : 6,84 pour ceux qui s'évaluent entre 9 et 10, contre 5,56 pour ceux qui s'évaluent à 5 ou moins. La relation est positive, mais les écarts d'auto-évaluation se traduisent par des écarts plus faibles dans les notes reçues.

*Précaution* : cette comparaison porte sur des moyennes individuelles et ne permet pas de mesurer précisément la capacité de chaque participant à s'évaluer. Les différences peuvent également refléter des écarts de sévérité dans la notation entre participants.

### 5.6 Premier ou dernier date de la soirée ?

On étudie le taux de « oui » du participant selon le numéro de la rencontre dans sa soirée. « Premier » désigne la première personne qu'il rencontre, « dernier » la dernière.

In [25]:
# Premier : order = 1
# Dernier : order = round (round est le nombre de partenaires rencontré)
df_dates["position"] = np.select(
    [df_dates["order"] == 1, df_dates["order"] == df_dates["round"]],
    ["Premier", "Dernier"], default="Milieu")

pos_rates = df_dates.groupby("position")["dec"].mean().reindex(["Premier", "Milieu", "Dernier"])
by_order = df_dates.groupby("order")["dec"].agg(["mean", "count"])
by_order = by_order[by_order["count"] >= 100]

fig = make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35],
                    subplot_titles=("Par numéro de rencontre", "Par position"))

fig.add_trace(go.Scatter(x=by_order.index, y=by_order["mean"] * 100,
    mode="lines+markers", line=dict(color=TINDER_PINK), name="Taux de oui"), row=1, col=1)

fig.add_trace(go.Bar(x=pos_rates.index, y=pos_rates.values * 100,
    marker_color=[TINDER_PINK, "#666666", TINDER_ORANGE],
    text=(pos_rates.values * 100).round(1).astype(str) + " %", textposition="inside",
    showlegend=False), row=1, col=2)

fig.update_layout(height=450, width=1000, title="Le taux de oui selon le moment de la soirée")

fig.update_xaxes(title_text="Numéro de rencontre", row=1, col=1)

fig.update_yaxes(title_text="Décisions positives (%)", row=1, col=1)

fig.update_yaxes(title_text="Décisions positives (%)", row=1, col=2)

fig.show()

fig.write_image(IMG_PATH / "10_ordre_rencontres.png")


**Interprétation**
- **Être le premier date semble être un avantage** : le taux de « oui » est plus élevé au début de la soirée que pour les rencontres du milieu.
- **Le taux pour les derniers rendez-vous est proche de celui du milieu de soirée.**
- Le taux de « oui » semble diminuer légèrement au fil de la soirée.

### 5.7 Origine commune ou intérêts communs ?

Ces deux variables décrivent la paire : on travaille sur `df_pairs`, une ligne par rencontre.

In [26]:
# Une ligne par paire : taux de match selon l'origine commune et les intérêts communs
race_rates = df_pairs.groupby("samerace")["match"].mean().rename(index={0: "Origines différentes", 1: "Même origine"})

pairs_corr = df_pairs.dropna(subset=["int_corr"]).copy()
pairs_corr["quartile"] = pd.qcut(pairs_corr["int_corr"], 4,
    labels=["Q1 (faibles)", "Q2", "Q3", "Q4 (forts)"])

corr_rates = pairs_corr.groupby("quartile", observed=True)["match"].mean()

print("Taux de match selon l'origine commune :")
print((100 * race_rates).round(1).astype(str) + " %")
print("\nTaux de match selon les quartiles de corrélation des intérêts :")
print((100 * corr_rates).round(1).astype(str) + " %")

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("Selon l'origine commune", "Selon la corrélation des intérêts"))

for col_idx, (table, title) in enumerate(((race_rates, "Origine"), (corr_rates, "Intérêts")), start=1):
    fig.add_trace(go.Bar(x=table.index.astype(str), y=table.values * 100,
        marker_color=TINDER_PINK, text=(table.values * 100).round(1).astype(str) + " %",
        textposition="inside", showlegend=False), row=1, col=col_idx)

fig.update_layout(height=450, width=1000, title="Le taux de match selon la ressemblance")

fig.update_yaxes(title_text="Matchs (%)", row=1, col=1)

fig.update_xaxes(tickangle=-20)

fig.show()

fig.write_image(IMG_PATH / "11_origine_vs_interets.png")

Taux de match selon l'origine commune :
samerace
Origines différentes    15.6 %
Même origine            18.0 %
Name: match, dtype: str

Taux de match selon les quartiles de corrélation des intérêts :
quartile
Q1 (faibles)    15.7 %
Q2              15.4 %
Q3              16.0 %
Q4 (forts)      18.8 %
Name: match, dtype: str


**Interprétation**  
L'origine commune présente un écart descriptif de 2,4 points dans le taux de match. La corrélation des intérêts déclarés présente une tendance positive pour les paires les plus proches, sans que sa significativité soit vérifiée. En revanche, les intérêts communs ressentis pendant la rencontre sont associés à des notes plus élevées lorsque le participant souhaite revoir son partenaire

## 6. Quelques tests statistiques simples

Deux types de tests statistiques sont utilisés pour compléter les analyses descriptives : le test du chi-deux pour étudier l'association entre le sexe et la décision, et le test de Mann–Whitney pour comparer les distributions des six critères de notation selon la décision. Le seuil retenu est de 5 %. Les rencontres d'une même personne ne sont pas indépendantes : les p-values doivent donc être interprétées avec prudence.

### 6.1 Le taux de « oui » varie-t-il selon le sexe ?


In [27]:
table_gender = pd.crosstab(df_dates["gender"], df_dates["dec"])

chi2_gender, p_gender, _, _ = stats.chi2_contingency(table_gender)

print(table_gender)
print(f"Test du chi-deux : statistique = {chi2_gender:.2f}, p-value = {p_gender:.3g}")

if p_gender < 0.05:
    print("Conclusion : association statistique entre le sexe et la décision dans cet échantillon.")
else:
    print("Conclusion : pas d'association statistiquement détectée au seuil de 5 %.")


dec        0     1
gender            
Femme   2114  1289
Homme   1852  1551
Test du chi-deux : statistique = 41.16, p-value = 1.4e-10
Conclusion : association statistique entre le sexe et la décision dans cet échantillon.


### 6.2 Les notes diffèrent-elles entre les « oui » et les « non » ?

On compare les notes des six critères à l'aide du test de Mann–Whitney. Le tableau donne aussi les moyennes afin de garder une lecture concrète des écarts. Les six critères sont étudiés ensemble dans cette comparaison descriptive ; les tests sont exploratoires et ne démontrent pas de causalité.


In [28]:
rows = []

for col in SCORE_COLS:
    oui = df_scored.loc[df_scored["dec"] == 1, col].dropna()
    non = df_scored.loc[df_scored["dec"] == 0, col].dropna()
    _, p_value = stats.mannwhitneyu(oui, non, alternative="two-sided")
    rows.append({"Critère": LABELS[col], "Moyenne si oui": oui.mean(),
                 "Moyenne si non": non.mean(), "Écart": oui.mean() - non.mean(),
                 "p-value>": p_value, "significant" : p_value<0.05})
score_tests = pd.DataFrame(rows).sort_values("Écart", ascending=False)
score_tests.round(3)


,Critère,Moyenne si oui,Moyenne si non,Écart,p-value>,significant
0,Attirance,7.242,5.348,1.894,0.0,True
5,Intérêts communs,6.442,4.638,1.804,0.0,True
3,Humour,7.315,5.657,1.657,0.0,True
1,Sincérité,7.565,6.812,0.753,0.0,True
2,Intelligence,7.721,7.025,0.695,0.0,True
4,Ambition,7.107,6.442,0.665,0.0,True


**Interprétation** : si les p-values sont inférieures à 0,05, les distributions de notes sont différentes entre les décisions « oui » et « non ». 

*Limite :* plusieurs tests sont effectués et les observations se répètent par participant. Les résultats doivent être considérés comme des indices descriptifs/inférentiels, pas comme des preuves causales.


## 7. Synthèse, recommandations et limites

### 7.1 Réponse à la problématique

| # | Question | Résultat | 
|---|---|---|
| Q1 | Quelle proportion de rencontres aboutit à un match ? Existe-t-il une différence entre les sexes ? |16,5 % des rencontres aboutissent à un match. Les hommes disent plus souvent « oui » que les femmes (45,6 % contre 37,9 %).|
| Q2 | Quels critères distinguent les rencontres qui plaisent ? | L'attirance présente le plus grand écart de notes, suivie des intérêts communs ressentis et de l'humour. Les écarts de notes sont plus faibles pour la sincérité, l'intelligence et l'ambition. |
| Q3 | Les préférences déclarées correspondent-elles aux perceptions de l'autre sexe ? |Pas complètement : l'intelligence et la sincérité sont surdéclarées, tandis que l'humour et les intérêts communs sont sous-déclarés. Chaque sexe surestime de 8 à 9 points le poids de l'attirance chez l'autre.|
| Q4 | Les participants savent-ils évaluer leur propre valeur ? | Les participants s'attribuent en moyenne des notes supérieures à celles qu'ils reçoivent, avec des écarts allant de 0,75 à 1,27 point. Pour l'attirance, les groupes qui s'auto-évaluent le mieux reçoivent également de meilleures notes en moyenne.|
| Q5 | Les intérêts communs et l'origine commune favorisent-ils les matchs ? | L'origine commune est associée descriptivement à un taux de match légèrement supérieur (18,0 % contre 15,6 %). Les intérêts communs déclarés présentent une tendance positive pour les paires les plus proches, sans que sa significativité soit vérifiée. Les intérêts communs ressentis pendant la rencontre sont associés à des notes plus élevées lorsque le participant souhaite revoir son partenaire.|
| Q6 | Le moment de la rencontre influence-t-il la décision ? | Le premier rendez-vous présente un taux de décisions positives supérieur à celui des rencontres du milieu de soirée. Le taux du dernier rendez-vous est proche de celui du milieu.  |

### 7.2 Recommandations pour l'équipe Tinder

1. **Valoriser ce qui crée l'attirance et l'humour dans le profil** (qualité des photos, bio qui reflète la personnalité) : ce sont les critères qui présentent les écarts de notes les plus importants entre les décisions positives et négatives.
2. **Faire ressentir les centres d'intérêt communs** (mise en avant des points communs dans l'interface) plutôt que de les calculer en arrière-plan : la similitude déclarée prédit mal les matchs, les notes de partage ressenti sont associées à une décision plus souvent positive.
3. **Éviter de supposer que l'autre sexe partage les mêmes priorités que celles qu'on lui attribue** : les perceptions diffèrent des préférences réellement déclarées, notamment pour l'attirance physique.
4. **Tester l'effet de l'ordre de présentation** : un profil vu en premier obtient plus de « oui ».

### 7.3 Limites

- **Échantillon** : étudiants d'une même université, en 2002 - 2004, dans un contexte de rencontre physique de 4 minutes. La transposition à une application de swipe est à prouver.
- **Effet de halo** : notes et décisions sont données au même moment, donc les associations ne sont pas des relations de cause à effet.
- **Indépendance** : les rencontres d'un même participant ne sont pas indépendantes ; les p-values peuvent être optimistes.
- **Données manquantes** : 15,6 % des rencontres n'ont pas les six notes et ont un taux de « oui » plus faible (36,6 % contre 42,7 %).
- **Périmètre** : les vagues 6 à 9 sont exclues
- **Déclaratif** : préférences et auto-évaluations sont déclarées, donc soumises aux biais de désirabilité sociale.

### 7.4 Pistes d'approfondissement

- Un modèle prédictif du match (arbres, forêts aléatoires) pour mesurer la capacité à prédire.
- Une analyse multiniveau pour tenir compte de la répétition des rencontres par participant.
- L'effet de la différence d'âge et de la vague (taille du groupe, règles de l'événement).